# From Routing to QAOA Evolution
## Complete DTU SCIQIS project reproduction notebook

This notebook follows the presentation order:

1. Research question  
2. Routing problem  
3. Routing → QUBO  
4. QUBO → Ising → QAOA  
5. Penalty-X vs Global-Grover mixers  
6. Cost function + COBYLA  
7. Results for $p=1,\ldots,110$  
8. Layer-by-layer quantum-state evolution  
9. Scientific-computing validation  
10. Takeaways and limitations  

The mechanism we want to make visible is

$$
\boxed{
\text{routing cost}
\rightarrow H_C
\rightarrow \text{phase}
\rightarrow H_M
\rightarrow \text{interference}
\rightarrow \text{probability}
}
$$

and the classical optimization loop

$$
(\boldsymbol\gamma,\boldsymbol\beta)
\rightarrow
|\psi(\boldsymbol\gamma,\boldsymbol\beta)\rangle
\rightarrow
\langle H_C\rangle
\rightarrow
\text{COBYLA}
\rightarrow
(\boldsymbol\gamma',\boldsymbol\beta').
$$

### Frozen depth-110 convention

This notebook replays the frozen depth sweep stored in `results/global_depth110/`:

- 14 edge qubits, $2^{14}=16384$ states,
- flow-penalty coefficient $A=6$,
- cost phase uses **raw penalized energy**,
- parameter order = all $\gamma$'s then all $\beta$'s,
- layer order = cost then mixer,
- Penalty-X uses $e^{-i\beta\sum_j X_j}$ **without** $\beta/n$ scaling,
- Global-Grover uses
  $$
  U_G(\beta)=I+(e^{-i\beta}-1)|s\rangle\langle s|.
  $$

By default the notebook reads the frozen results and exactly replays the saved $p=110$ optimized circuits.  
A final optional section can rerun the complete optimizer sweep.

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
from pathlib import Path
import sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from scipy.optimize import minimize, Bounds
from scipy.sparse import coo_matrix

warnings.filterwarnings("ignore", category=RuntimeWarning)

def find_repo_root():
    """Find sciqis-qaoa-routing from the current notebook location."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "data" / "graph.json").exists()
            and (candidate / "src" / "qaoa.py").exists()
            and (candidate / "results" / "global_depth110" / "depth_by_depth.csv").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from inside "
        "sciqis-qaoa-routing/ or its notebooks/ directory."
    )

ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from graph import (
    load_graph, get_edge_order, exact_route,
    path_cost, path_to_edge_bitstring
)
from qubo import (
    build_qubo, qubo_to_ising, enumerate_state_space,
    incidence_matrix, node_supplies, max_qubo_ising_error
)
from qaoa import (
    initial_state, initial_parameters,
    apply_cost, apply_x_mixer, apply_grover_mixer,
    probabilities
)
from metrics import distribution_metrics, top_state_rows, shannon_entropy

RESULT_ROOT = ROOT / "results" / "global_depth110"
OUTPUT_ROOT = ROOT / "notebook_output" / "qaoa_project_reproduction"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root :", ROOT)
print("Frozen results  :", RESULT_ROOT)
print("Figure outputs  :", OUTPUT_ROOT)

# 1. Research Question

> **How does QAOA transform probability through repeated cost and mixer evolution, and how does this behavior differ between Penalty-X and Global-Grover QAOA as depth increases from $p=1$ to $p=110$?**

The notebook explicitly inspects the Hamiltonians, statevectors, phases, probabilities, optimizer objective, and every optimized QAOA layer rather than treating QAOA as a black box.

# 2. Routing Problem

One binary variable is assigned to every directed edge:

$$
x_e\in\{0,1\}.
$$

The route objective is

$$
C(x)=\sum_e w_e x_e.
$$

Flow conservation is

$$
\sum_{e\in\mathrm{out}(v)}x_e
-
\sum_{e\in\mathrm{in}(v)}x_e
=
b_v,
$$

where $b_s=1$, $b_t=-1$, and $b_v=0$ elsewhere.

In [ ]:
# 2.1 Load graph and exact classical reference
graph = load_graph(ROOT / "data" / "graph.json")
edge_order = get_edge_order(graph)
optimal_route, optimal_cost = exact_route(graph)

print(f"Nodes            : {graph.number_of_nodes()}")
print(f"Edges / qubits   : {graph.number_of_edges()}")
print(f"Source -> target : {graph.graph['source']} -> {graph.graph['target']}")
print(f"Exact route      : {' -> '.join(map(str, optimal_route))}")
print(f"Exact cost       : {optimal_cost}")
print(f"State-space size : 2^{len(edge_order)} = {2**len(edge_order):,}")

edge_table = pd.DataFrame([
    {
        "qubit": f"q{i}",
        "variable": f"x_{i}",
        "edge": f"{u} -> {v}",
        "weight": graph.edges[u, v]["weight"],
    }
    for i, (u, v) in enumerate(edge_order)
])
display(edge_table)

optimal_bits = path_to_edge_bitstring(graph, optimal_route)
print("Optimal route bitstring (q0 -> q13):", "".join(map(str, optimal_bits)))

In [ ]:
# 2.2 Plot fixed graph and highlight exact shortest path
pos = nx.spring_layout(graph, seed=11)
best_edges = set(zip(optimal_route, optimal_route[1:]))

plt.figure(figsize=(10, 6))
nx.draw_networkx_nodes(graph, pos, node_size=900)
nx.draw_networkx_labels(graph, pos)
nx.draw_networkx_edges(graph, pos, arrows=True, arrowsize=18, width=1.2)
nx.draw_networkx_edges(
    graph, pos, edgelist=list(best_edges),
    arrows=True, arrowsize=20, width=3.0
)
labels = {(u, v): graph.edges[u, v]["weight"] for u, v in graph.edges()}
nx.draw_networkx_edge_labels(graph, pos, edge_labels=labels)
plt.title(f"Fixed routing graph — exact optimum cost = {optimal_cost}")
plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "01_graph_optimal_route.png", dpi=180, bbox_inches="tight")
plt.show()

# 3. Routing → QUBO

Invalid collections of cheap edges can have a low raw routing cost.  
The flow penalty is therefore

$$
P_{\mathrm{flow}}(x)
=
\sum_v
\left(
\sum_{\mathrm{out}(v)}x_e
-
\sum_{\mathrm{in}(v)}x_e
-
b_v
\right)^2.
$$

The frozen project objective is

$$
\boxed{
Q(x)=
\sum_e w_e x_e
+
6P_{\mathrm{flow}}(x)
}.
$$

In [ ]:
# 3.1 Enumerate all 16,384 states and build the QUBO
PENALTY = 6.0

states = enumerate_state_space(graph)
qubo = build_qubo(graph, PENALTY)
ising = qubo_to_ising(qubo)

routing_costs = np.array([s.routing_cost for s in states], dtype=float)
flow_penalties = np.array([s.flow_penalty for s in states], dtype=float)
raw_energies = routing_costs + PENALTY * flow_penalties

feasible_mask = np.array([s.is_decoder_valid for s in states], dtype=bool)
optimal_mask = np.array(
    [s.is_decoder_valid and s.routing_cost == optimal_cost for s in states],
    dtype=bool,
)

print("All states      :", len(states))
print("Feasible states :", int(feasible_mask.sum()))
print("Optimal states  :", int(optimal_mask.sum()))
print("Energy range    :", raw_energies.min(), "to", raw_energies.max())

In [ ]:
# 3.2 Flow incidence matrix
B = np.array(incidence_matrix(graph), dtype=int)
incidence_df = pd.DataFrame(
    B,
    index=[f"node {v}" for v in sorted(graph.nodes())],
    columns=[f"q{i}" for i in range(len(edge_order))]
)
display(incidence_df)
display(pd.Series(node_supplies(graph), name="b_v"))

In [ ]:
# 3.3 QUBO coefficients
print("QUBO constant =", float(qubo.constant))

display(pd.DataFrame({
    "qubit": [f"q{i}" for i in range(len(qubo.linear))],
    "linear coefficient": [float(c) for c in qubo.linear],
}))

pair_df = pd.DataFrame([
    {"term": f"x_{i} x_{j}", "i": i, "j": j, "coefficient": float(c)}
    for (i, j), c in sorted(qubo.pair.items())
])
display(pair_df)

In [ ]:
# 3.4 Three-dimensional energy landscape over all computational-basis states
# The 14-bit basis index is split into two 7-bit coordinates only for visualization.
# The vertical coordinate remains the exact, unsmoothed penalized QUBO energy Q(x).
idx = np.arange(len(states), dtype=np.int64)
n_bits = len(edge_order)
n_low = n_bits // 2
n_high = n_bits - n_low
high_coord = idx >> n_low
low_coord = idx & ((1 << n_low) - 1)

grid_shape = (1 << n_high, 1 << n_low)
X = high_coord.reshape(grid_shape)
Y = low_coord.reshape(grid_shape)
Z = raw_energies.reshape(grid_shape)
energy_span = float(np.ptp(raw_energies))
z_floor = float(raw_energies.min() - 0.08 * max(energy_span, 1.0))

with plt.style.context("seaborn-v0_8-whitegrid"):
    fig = plt.figure(figsize=(14, 8), facecolor="#f7f9fc")
    ax = fig.add_subplot(111, projection="3d", computed_zorder=False)

    surface = ax.plot_surface(
        X, Y, Z,
        cmap="viridis",
        rcount=grid_shape[0], ccount=grid_shape[1],
        linewidth=0, antialiased=True, alpha=0.82, zorder=1,
    )
    ax.contourf(
        X, Y, Z, zdir="z", offset=z_floor,
        levels=24, cmap="viridis", alpha=0.32, zorder=0,
    )

    # Overlay the exact feasible routes and the unique optimum.
    ax.scatter(
        high_coord[feasible_mask], low_coord[feasible_mask], raw_energies[feasible_mask],
        s=52, c="#42e8e0", edgecolors="#082f49", linewidths=0.8,
        depthshade=False, label=f"feasible routes ({feasible_mask.sum()})", zorder=5,
    )
    ax.scatter(
        high_coord[optimal_mask], low_coord[optimal_mask], raw_energies[optimal_mask],
        s=260, c="#ffd166", edgecolors="#7c2d12", linewidths=1.5,
        marker="*", depthshade=False, label="optimal route", zorder=8,
    )

    # q0 is the least-significant state-index bit in this repository.
    ax.set_xlabel(fr"high-order edge bits $x_{{{n_low}}},\ldots,x_{{{n_bits - 1}}}$", labelpad=12)
    ax.set_ylabel(fr"low-order edge bits $x_0,\ldots,x_{{{n_low - 1}}}$", labelpad=12)
    ax.set_zlabel(r"penalized energy $Q(x)$", labelpad=12)
    ax.set_zlim(z_floor, float(raw_energies.max()))
    ax.view_init(elev=29, azim=-132)
    ax.set_box_aspect((1.25, 1.0, 0.72))

    ax.set_title(
        rf"QUBO energy landscape across all $2^{{{n_bits}}}={len(states):,}$ basis states"
        "\nExact energies; feasible routes and the optimum are highlighted",
        fontsize=17, fontweight="bold", pad=22, color="#102a43",
    )
    ax.legend(loc="upper left", bbox_to_anchor=(0.01, 0.97), frameon=True, framealpha=0.95)
    cbar = fig.colorbar(surface, ax=ax, shrink=0.64, pad=0.08, aspect=20)
    cbar.set_label(r"penalized energy $Q(x)$", rotation=90, labelpad=13)

    for pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
        pane.set_facecolor((0.94, 0.97, 1.00, 0.72))
        pane.set_edgecolor((0.72, 0.79, 0.88, 0.85))
    ax.tick_params(colors="#334e68", labelsize=9)

    plt.savefig(OUTPUT_ROOT / "02_energy_landscape.png", dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

# 4. QUBO → Ising → Cost Hamiltonian

Using

$$
x_j=\frac{I-Z_j}{2},
$$

the cost Hamiltonian is

$$
\boxed{
H_C=c_0I+\sum_j h_jZ_j+\sum_{j<k}J_{jk}Z_jZ_k
}.
$$

Because it is diagonal in the computational basis,

$$
H_C|z\rangle=E_z|z\rangle.
$$

For 14 qubits,

$$
H_C\in\mathbb C^{16384\times16384},
$$

but the numerical implementation only needs its diagonal vector of 16,384 energies.

In [ ]:
# 4.1 Print Ising terms and interaction matrix
print("Ising constant c0 =", float(ising.constant))

display(pd.DataFrame({
    "term": [f"Z_{i}" for i in range(len(ising.h))],
    "h_i": [float(c) for c in ising.h],
}))

J = np.zeros((len(edge_order), len(edge_order)))
for (i, j), c in ising.coupling.items():
    J[i, j] = J[j, i] = float(c)

# Presentation-quality lower-triangular coupling map.
# J is symmetric, so the upper triangle is redundant; zero couplings are hidden.
from matplotlib.colors import TwoSlopeNorm

n_qubits = len(edge_order)
strict_lower = np.tril(np.ones_like(J, dtype=bool), k=-1)
visible = strict_lower & ~np.isclose(J, 0.0)
J_lower = np.ma.masked_where(~visible, J)

max_abs_coupling = float(np.max(np.abs(J[visible]))) if np.any(visible) else 1.0
norm = TwoSlopeNorm(vmin=-max_abs_coupling, vcenter=0.0, vmax=max_abs_coupling)
cmap = plt.colormaps["RdBu_r"].copy()
cmap.set_bad("white")

with plt.style.context("seaborn-v0_8-whitegrid"):
    fig, ax = plt.subplots(figsize=(9.2, 8.2), facecolor="white")
    im = ax.imshow(
        J_lower, cmap=cmap, norm=norm, interpolation="none",
        origin="upper", aspect="equal",
    )

    labels = [f"q{i}" for i in range(n_qubits)]
    ax.set_xticks(range(n_qubits), labels, rotation=0)
    ax.set_yticks(range(n_qubits), labels)
    ax.set_xlabel(r"qubit $j$", labelpad=10)
    ax.set_ylabel(r"qubit $i$", labelpad=10)

    # Keep every matrix element visually square and separate cells cleanly.
    ax.set_xticks(np.arange(-0.5, n_qubits, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_qubits, 1), minor=True)
    ax.grid(which="minor", color="#d9e2ec", linewidth=0.75)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(axis="both", labelsize=10, colors="#243b53")

    # Label every non-zero interaction with its exact stored coefficient.
    for i, j in zip(*np.where(visible)):
        value = float(J[i, j])
        label = f"{value:.0f}" if np.isclose(value, round(value)) else f"{value:.2g}"
        text_color = "white" if abs(value) >= 0.48 * max_abs_coupling else "#102a43"
        ax.text(j, i, label, ha="center", va="center",
                fontsize=8.8, fontweight="bold", color=text_color)

    ax.set_title(
        r"Ising coupling matrix $J_{ij}$"
        "\nStrict lower triangle with non-zero interactions only",
        fontsize=16, fontweight="bold", color="#102a43", pad=16,
    )
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.045)
    cbar.set_label(r"coupling coefficient $J_{ij}$", rotation=90, labelpad=12)
    cbar.ax.axhline(0, color="#243b53", linewidth=1.0)

    fig.text(
        0.5, 0.025,
        r"White cells denote $J_{ij}=0$ or the redundant symmetric half.",
        ha="center", fontsize=10, color="#486581",
    )
    fig.tight_layout(rect=(0.0, 0.055, 1.0, 1.0))
    plt.savefig(OUTPUT_ROOT / "03_ising_coupling_matrix.png", dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()

In [ ]:
# 4.2 Validate QUBO and Ising on every basis state
ising_error = float(max_qubo_ising_error(states, [qubo]))
qubo_energy = np.array([float(qubo.evaluate(s.edge_vector)) for s in states])
ising_energy = np.array([float(ising.basis_energy(s.edge_vector)) for s in states])

assert np.allclose(qubo_energy, raw_energies)
assert np.allclose(ising_energy, raw_energies)
assert ising_error == 0.0

print("PASS: QUBO = Ising = routing cost + 6 × flow penalty for all 16,384 states.")

In [ ]:
# 4.3 Complete computational-basis energy record
# Record every exact basis-state eigenvalue E_x = Q(x); do not construct or plot a dense matrix.
basis_energy_table = pd.DataFrame({
    "basis_index": [state.state_index for state in states],
    # Project convention: edge variables are written from q0 to q13.
    "edge_bits_q0_to_q13": [state.canonical_bitstring for state in states],
    # Standard integer-index ket display: most-significant bit q13 appears first.
    "basis_ket_q13_to_q0": [f"|{state.canonical_bitstring[::-1]}>" for state in states],
    "energy_E_x_equals_Q_x": raw_energies.astype(float),
    "routing_cost_C_x": routing_costs.astype(float),
    "flow_penalty_P_flow_x": flow_penalties.astype(float),
    "feasible": feasible_mask,
    "optimal": optimal_mask,
    "decoded_route": [
        " -> ".join(map(str, state.decoded_route)) if state.decoded_route else ""
        for state in states
    ],
})

# Independent consistency checks before saving the complete record.
assert len(basis_energy_table) == 2 ** len(edge_order)
assert np.array_equal(basis_energy_table["basis_index"].to_numpy(), np.arange(len(states)))
assert np.allclose(basis_energy_table["energy_E_x_equals_Q_x"], raw_energies)
assert int(basis_energy_table["feasible"].sum()) == int(feasible_mask.sum())
assert int(basis_energy_table["optimal"].sum()) == int(optimal_mask.sum())

energy_record_path = OUTPUT_ROOT / "04_all_basis_state_energies.csv"
basis_energy_table.to_csv(energy_record_path, index=False, float_format="%.12g")

print(f"Recorded basis states : {len(basis_energy_table):,}")
print(f"Energy range          : {raw_energies.min():.6g} to {raw_energies.max():.6g}")
print(f"Feasible / optimal    : {feasible_mask.sum()} / {optimal_mask.sum()}")
print(f"Complete CSV          : {energy_record_path}")
print("\nPreview — first five states:")
display(basis_energy_table.head(5))
print("Preview — all feasible states:")
display(basis_energy_table.loc[basis_energy_table["feasible"]].reset_index(drop=True))

# 5. Two QAOA Designs

For depth $p$,

$$
|\psi_p\rangle
=
\prod_{\ell=1}^p
U_M(\beta_\ell)
U_C(\gamma_\ell)
|\psi_0\rangle,
\qquad
|\psi_0\rangle=|+\rangle^{\otimes 14}.
$$

## Penalty-X mixer

$$
H_M^{(X)}=\sum_jX_j,
\qquad
U_M^{(X)}(\beta)=e^{-i\beta H_M^{(X)}}.
$$

It connects bitstrings with Hamming distance 1.

## Global-Grover mixer

$$
H_M^{(G)}=|s\rangle\langle s|,
$$

$$
U_M^{(G)}(\beta)
=
I+(e^{-i\beta}-1)|s\rangle\langle s|.
$$

This is dense but rank one, so the code never constructs a $16384\times16384$ dense matrix.

In [ ]:
# Compact QAOA circuit schematic: show only four layers
from matplotlib.patches import FancyBboxPatch

display_layers = 4
wire_y = np.arange(n_qubits)[::-1]
top_y = float(wire_y.max()) + 0.45
bottom_y = float(wire_y.min()) - 0.45
middle_y = (top_y + bottom_y) / 2
block_height = top_y - bottom_y
block_width = 0.58
layer_start = 1.75
layer_stride = 1.55
within_layer_gap = 0.17
final_mixer_edge = layer_start + (display_layers - 1) * layer_stride + 2 * block_width + within_layer_gap
measure_x = final_mixer_edge + 1.25

cost_color = "#6c5ce7"
mixer_color = "#159d9c"
wire_color = "#78909c"
text_color = "#102a43"

with plt.style.context("seaborn-v0_8-white"):
    fig, ax = plt.subplots(figsize=(14.5, 7.6), facecolor="white")

    # Fourteen wires correspond to the fourteen directed-edge variables.
    for qubit, y in enumerate(wire_y):
        ax.plot([0.25, final_mixer_edge], [y, y], color=wire_color, linewidth=1.15, zorder=1)
        ax.plot(
            [final_mixer_edge, measure_x], [y, y], color=wire_color,
            linewidth=1.15, linestyle=(0, (2, 3)), zorder=1,
        )
        ax.plot([measure_x + 0.62, measure_x + 0.95], [y, y], color=wire_color, linewidth=1.15, zorder=1)
        ax.text(0.08, y, rf"$q_{{{qubit}}}$", ha="right", va="center", fontsize=9.5, color=text_color)

        # H|0> = |+>: uniform full-space initialization.
        gate = FancyBboxPatch(
            (0.58, y - 0.20), 0.40, 0.40,
            boxstyle="round,pad=0.02,rounding_size=0.04",
            facecolor="#eef3f7", edgecolor="#486581", linewidth=1.0, zorder=3,
        )
        ax.add_patch(gate)
        ax.text(0.78, y, r"$H$", ha="center", va="center", fontsize=9, color=text_color, zorder=4)

    ax.text(0.78, top_y + 0.32, r"prepare $|+\rangle^{\otimes14}$", ha="center", va="bottom", fontsize=10, color=text_color)

    # Each colored block acts on all 14 qubits; only four repeated layers are expanded.
    for layer in range(1, display_layers + 1):
        cost_x = layer_start + (layer - 1) * layer_stride
        mixer_x = cost_x + block_width + within_layer_gap

        cost_box = FancyBboxPatch(
            (cost_x, bottom_y), block_width, block_height,
            boxstyle="round,pad=0.03,rounding_size=0.08",
            facecolor=cost_color, edgecolor="#4933a9", linewidth=1.1, alpha=0.94, zorder=2,
        )
        mixer_box = FancyBboxPatch(
            (mixer_x, bottom_y), block_width, block_height,
            boxstyle="round,pad=0.03,rounding_size=0.08",
            facecolor=mixer_color, edgecolor="#087f7e", linewidth=1.1, alpha=0.94, zorder=2,
        )
        ax.add_patch(cost_box)
        ax.add_patch(mixer_box)
        ax.text(
            cost_x + block_width / 2, middle_y,
            "$U_C$\n" + rf"$\gamma_{{{layer}}}$",
            ha="center", va="center", color="white", fontsize=11, fontweight="bold", zorder=4,
        )
        ax.text(
            mixer_x + block_width / 2, middle_y,
            "$U_M$\n" + rf"$\beta_{{{layer}}}$",
            ha="center", va="center", color="white", fontsize=11, fontweight="bold", zorder=4,
        )
        ax.text(
            (cost_x + mixer_x + block_width) / 2, top_y + 0.32, f"layer {layer}",
            ha="center", va="bottom", fontsize=10, fontweight="bold", color=text_color,
        )

    ax.text(
        (final_mixer_edge + measure_x) / 2, middle_y, r"$\cdots$",
        ha="center", va="center", fontsize=28, color="#486581",
        bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.15}, zorder=4,
    )

    measure_box = FancyBboxPatch(
        (measure_x, bottom_y), 0.62, block_height,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        facecolor="#f4c95d", edgecolor="#b8860b", linewidth=1.1, alpha=0.96, zorder=2,
    )
    ax.add_patch(measure_box)
    ax.text(
        measure_x + 0.31, middle_y, "measure", rotation=90,
        ha="center", va="center", fontsize=10, fontweight="bold", color="#5d4300", zorder=4,
    )

    ax.text(
        (layer_start + final_mixer_edge) / 2, bottom_y - 0.55,
        r"purple: $U_C(\gamma_\ell)=e^{-i\gamma_\ell H_C}$    ·    teal: $U_M(\beta_\ell)$",
        ha="center", va="top", fontsize=10.5, color=text_color,
    )
    ax.text(
        (measure_x + 0.95) / 2, bottom_y - 1.02,
        r"Only four layers are drawn; the frozen depth study repeats the same cost–mixer pattern through $p=110$.",
        ha="center", va="top", fontsize=10, color="#486581",
    )

    ax.set_xlim(-0.55, measure_x + 1.05)
    ax.set_ylim(bottom_y - 1.35, top_y + 1.05)
    ax.set_title(
        "QAOA circuit anatomy — first four layers shown",
        fontsize=16, fontweight="bold", color=text_color, pad=18,
    )
    ax.axis("off")
    plt.savefig(
        OUTPUT_ROOT / "05_qaoa_circuit_four_layers.png",
        dpi=200, bbox_inches="tight", facecolor="white",
    )
    plt.show()

In [ ]:
# 5.1 Explicit small mixer matrices + actual X-mixer sparsity
def dense_x_hamiltonian(n):
    d = 1 << n
    H = np.zeros((d, d))
    for z in range(d):
        for q in range(n):
            H[z, z ^ (1 << q)] += 1
    return H

def dense_grover_projector(n):
    d = 1 << n
    return np.ones((d, d)) / d

print("3-qubit X mixer")
display(pd.DataFrame(dense_x_hamiltonian(3).astype(int)))

print("3-qubit Global-Grover projector")
display(pd.DataFrame(dense_grover_projector(3)))

n_qubits = len(edge_order)
dim = 1 << n_qubits
rows, cols, data = [], [], []
for z in range(dim):
    for q in range(n_qubits):
        rows.append(z)
        cols.append(z ^ (1 << q))
        data.append(1.0)

Hx = coo_matrix((data, (rows, cols)), shape=(dim, dim)).tocsr()

x_density = Hx.nnz / (dim * dim)
grover_density = 1.0
print(f"{n_qubits}-qubit X mixer nonzeros:", Hx.nnz)
print(f"{n_qubits}-qubit X mixer density :", x_density)
print("Grover projector entry       :", 1 / dim)

# Render a binary raster instead of thousands of tiny spy markers. The first
# 512 states form a 9-dimensional subcube, hence nine visible neighbors per row.
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyBboxPatch, Patch

corner_size = min(512, dim)
corner = Hx[:corner_size, :corner_size].toarray().astype(bool)
visible_degree = int(corner[0].sum())
density_ratio = grover_density / x_density

navy = "#102a43"
slate = "#627d98"
grid = "#d9e2ec"
paper = "#f7f9fc"
x_color = "#5b5fef"
grover_color = "#f59e0b"

with plt.style.context("seaborn-v0_8-whitegrid"):
    fig = plt.figure(figsize=(13.6, 7.6), facecolor=paper)
    outer = fig.add_gridspec(
        1, 2, width_ratios=(1.62, 1.0), wspace=0.20,
        left=0.06, right=0.96, bottom=0.105, top=0.82,
    )

    # A | Exact sparsity pattern
    ax_matrix = fig.add_subplot(outer[0])
    binary_cmap = ListedColormap([paper, x_color])
    ax_matrix.imshow(
        corner, cmap=binary_cmap, interpolation="nearest",
        origin="upper", aspect="equal", rasterized=True,
    )
    for boundary in np.arange(64, corner_size, 64):
        ax_matrix.axhline(boundary - 0.5, color="white", linewidth=0.45, alpha=0.82)
        ax_matrix.axvline(boundary - 0.5, color="white", linewidth=0.45, alpha=0.82)
    matrix_ticks = np.linspace(0, corner_size - 1, 5, dtype=int)
    ax_matrix.set_xticks(matrix_ticks)
    ax_matrix.set_yticks(matrix_ticks)
    ax_matrix.set_xlabel(r"basis-state index $j$", color=navy, labelpad=9)
    ax_matrix.set_ylabel(r"basis-state index $i$", color=navy, labelpad=9)
    ax_matrix.set_title(
        rf"A  |  ${corner_size}\times{corner_size}$ principal block",
        loc="left", color=navy, fontsize=13, fontweight="bold", pad=12,
    )
    ax_matrix.tick_params(colors=slate, labelsize=9)
    ax_matrix.grid(False)
    for spine in ax_matrix.spines.values():
        spine.set_color(grid)
    ax_matrix.legend(
        handles=[
            Patch(facecolor=x_color, edgecolor="none", label="non-zero coupling"),
            Patch(facecolor=paper, edgecolor=grid, label="zero"),
        ],
        loc="upper right", frameon=True, facecolor="white",
        edgecolor=grid, framealpha=0.96, fontsize=9,
    )

    # B | Structural summary and the sparse-vs-dense comparison
    right = outer[1].subgridspec(2, 1, height_ratios=(1.08, 0.92), hspace=0.34)
    ax_cards = fig.add_subplot(right[0])
    ax_cards.set_xlim(0, 1)
    ax_cards.set_ylim(0, 1)
    ax_cards.axis("off")
    ax_cards.set_title(
        "B  |  Full X-mixer matrix", loc="left",
        color=navy, fontsize=13, fontweight="bold", pad=10,
    )

    cards = [
        (0.00, 0.53, rf"$2^{{{n_qubits}}}$", "Hilbert-space dimension"),
        (0.52, 0.53, f"{n_qubits}", "nonzeros per row"),
        (0.00, 0.05, f"{Hx.nnz:,}", "nonzeros in total"),
        (0.52, 0.05, f"{100 * x_density:.4f}%", "matrix density"),
    ]
    for x, y, value, label in cards:
        card = FancyBboxPatch(
            (x, y), 0.46, 0.36,
            boxstyle="round,pad=0.018,rounding_size=0.025",
            facecolor="white", edgecolor=grid, linewidth=1.0,
            transform=ax_cards.transAxes,
        )
        ax_cards.add_patch(card)
        ax_cards.text(
            x + 0.04, y + 0.22, value, transform=ax_cards.transAxes,
            color=x_color, fontsize=18, fontweight="bold", va="center",
        )
        ax_cards.text(
            x + 0.04, y + 0.085, label, transform=ax_cards.transAxes,
            color=slate, fontsize=8.8, va="center",
        )

    ax_density = fig.add_subplot(right[1])
    density_values = [x_density, grover_density]
    density_y = [1, 0]
    density_colors = [x_color, grover_color]
    density_labels = [r"X mixer  $\sum_q X_q$", r"Grover  $|+\rangle\langle+|$"]
    for y, value, color in zip(density_y, density_values, density_colors):
        ax_density.hlines(y, 3e-4, value, color=color, linewidth=7, alpha=0.88)
        ax_density.scatter(value, y, s=62, color=color, edgecolor="white", zorder=3)
    ax_density.set_xscale("log")
    ax_density.set_xlim(3e-4, 1.55)
    ax_density.set_ylim(-0.55, 1.55)
    ax_density.set_yticks(density_y, density_labels)
    ax_density.set_xlabel("fraction of non-zero matrix entries", color=navy, labelpad=8)
    ax_density.set_title(
        "C  |  Matrix-density comparison", loc="left",
        color=navy, fontsize=13, fontweight="bold", pad=12,
    )
    ax_density.text(
        x_density * 1.18, 1, f"{100 * x_density:.4f}%",
        color=x_color, fontsize=9.5, fontweight="bold", va="center",
    )
    ax_density.text(
        0.88, 0.17, "100%", transform=ax_density.transAxes,
        color=grover_color, fontsize=9.5, fontweight="bold", ha="right",
    )
    ax_density.text(
        0.98, 0.94, f"Grover is {density_ratio:,.0f}× denser",
        transform=ax_density.transAxes, ha="right", va="top",
        color=navy, fontsize=9.5,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=grid),
    )
    ax_density.grid(axis="x", color=grid, linewidth=0.8)
    ax_density.grid(axis="y", visible=False)
    ax_density.tick_params(axis="x", colors=slate, labelsize=8.5)
    ax_density.tick_params(axis="y", colors=navy, labelsize=9.2, length=0, pad=7)
    for side in ("top", "right", "left"):
        ax_density.spines[side].set_visible(False)
    ax_density.spines["bottom"].set_color(grid)

    fig.suptitle(
        "Local X mixing is sparse; global Grover mixing is dense",
        x=0.06, y=0.955, ha="left", color=navy,
        fontsize=18, fontweight="bold",
    )
    fig.text(
        0.06, 0.895,
        rf"Each computational-basis state connects to exactly {n_qubits} Hamming-distance-one neighbors; "
        "the Grover projector couples every pair of states.",
        ha="left", color=slate, fontsize=10.5,
    )
    fig.text(
        0.06, 0.035,
        rf"The displayed principal block contains {visible_degree} in-block neighbors per row; "
        rf"the remaining {n_qubits - visible_degree} couplings leave the block.",
        ha="left", color=slate, fontsize=9.2,
    )

    plt.savefig(
        OUTPUT_ROOT / "05_x_mixer_sparsity.png",
        dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor(),
    )
    plt.show()

## 5.2 Cost Hamiltonian: energy → phase

If

$$
|\psi\rangle=\sum_z\alpha_z|z\rangle,
$$

then

$$
U_C(\gamma)|\psi\rangle
=
\sum_z\alpha_ze^{-i\gamma E_z}|z\rangle.
$$

Therefore

$$
|\alpha_ze^{-i\gamma E_z}|^2=|\alpha_z|^2.
$$

So

$$
\boxed{\text{the cost layer changes phase, not basis probability.}}
$$

The mixer then couples amplitudes and converts those relative phase differences into constructive/destructive interference.

In [ ]:
# 5.2 Demonstrate cost-phase invariance and mixer redistribution
gamma_demo = 0.73
beta_demo = 0.61

psi0 = initial_state(n_qubits)
p0 = probabilities(psi0)

psi_cost = apply_cost(psi0, raw_energies, gamma_demo)
p_cost = probabilities(psi_cost)

cost_probability_change = np.max(np.abs(p_cost - p0))
print("Max probability change after COST:", cost_probability_change)
assert cost_probability_change < 1e-12

phase_shift = np.angle(psi_cost / psi0)
sample = np.argsort(raw_energies)[::45]

plt.figure(figsize=(9, 5))
plt.scatter(raw_energies[sample], phase_shift[sample], s=14)
plt.xlabel(r"energy $E_z$")
plt.ylabel("phase [rad]")
plt.title(r"Cost phase: $\Delta\phi_z=-\gamma E_z$ modulo $2\pi$")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "06_energy_to_phase.png", dpi=180, bbox_inches="tight")
plt.show()

psi_x = apply_x_mixer(psi_cost, beta_demo, n_qubits)
psi_g = apply_grover_mixer(psi_cost, beta_demo)
p_x = probabilities(psi_x)
p_g = probabilities(psi_g)

print("Max probability change after X mixer     :", np.max(np.abs(p_x - p_cost)))
print("Max probability change after Grover mixer:", np.max(np.abs(p_g - p_cost)))

In [ ]:
# 5.3 Probability changes occur after mixer, not after cost
top_change = np.argsort(np.maximum(np.abs(p_x-p_cost), np.abs(p_g-p_cost)))[-80:]

plt.figure(figsize=(11, 4))
plt.plot(top_change, (p_x-p_cost)[top_change], marker="o", linestyle="")
plt.axhline(0)
plt.xlabel("selected basis-state index")
plt.ylabel("probability change")
plt.title("Penalty-X mixer probability redistribution")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "07_x_probability_redistribution.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 4))
plt.plot(top_change, (p_g-p_cost)[top_change], marker="o", linestyle="")
plt.axhline(0)
plt.xlabel("selected basis-state index")
plt.ylabel("probability change")
plt.title("Global-Grover mixer probability redistribution")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "08_grover_probability_redistribution.png", dpi=180, bbox_inches="tight")
plt.show()

# 6. Cost Function and COBYLA

The main sweep minimizes

$$
f(\boldsymbol\gamma,\boldsymbol\beta)
=
\langle\psi_p|H_C|\psi_p\rangle
=
\sum_zP_zE_z.
$$

COBYLA does **not** change $H_C$ or $H_M$.  
It changes only

$$
\gamma_1,\ldots,\gamma_p,\beta_1,\ldots,\beta_p.
$$

The loop is:

1. propose $\theta$,
2. simulate QAOA,
3. compute $P(z)$,
4. return one scalar expectation value,
5. update $\theta$,
6. repeat.

The main metrics are

$$
P_{\rm feas},\qquad
P_{\rm opt},\qquad
P_{\rm opt\mid feas}
=
\frac{P_{\rm opt}}{P_{\rm feas}}.
$$

In [ ]:
# 6.1 Exact simulator matching the frozen p=1..110 convention
def split_parameters(theta, depth):
    theta = np.asarray(theta, dtype=float)
    assert theta.shape == (2 * depth,)
    return theta[:depth], theta[depth:]

def simulate_full_space(theta, depth, mixer):
    gammas, betas = split_parameters(theta, depth)
    state = initial_state(n_qubits)

    for gamma, beta in zip(gammas, betas):
        state = apply_cost(state, raw_energies, gamma)

        if mixer == "penalty_x":
            # Frozen global_depth110 convention: NO beta/n scaling.
            state = apply_x_mixer(state, beta, n_qubits)
        elif mixer == "global_grover":
            state = apply_grover_mixer(state, beta)
        else:
            raise ValueError(mixer)

    return state

def state_summary(state):
    probs = probabilities(state)
    metrics = distribution_metrics(probs, states, optimal_cost=optimal_cost)
    return {
        "expected_hc": float(probs @ raw_energies),
        "p_feas": metrics.p_feas,
        "p_opt": metrics.p_opt,
        "p_opt_given_feasible": metrics.p_opt_given_feas,
        "entropy": shannon_entropy(probs),
    }

def objective(theta, depth, mixer):
    probs = probabilities(simulate_full_space(theta, depth, mixer))
    return float(probs @ raw_energies)

In [ ]:
# 6.2 Small live COBYLA demonstration at p=1
RUN_COBYLA_DEMO = True
DEMO_BUDGET = 80
cobyla_rows = []

if RUN_COBYLA_DEMO:
    depth = 1
    mixer = "penalty_x"
    seed = 2601 + 7919 * depth
    start = initial_parameters(depth, seed)

    def logged_objective(theta):
        value = objective(theta, depth, mixer)
        s = state_summary(simulate_full_space(theta, depth, mixer))
        best = min([r["objective"] for r in cobyla_rows], default=np.inf)
        cobyla_rows.append({
            "evaluation": len(cobyla_rows) + 1,
            "gamma_1": theta[0],
            "beta_1": theta[1],
            "objective": value,
            "best_so_far": min(best, value),
            "p_feas": s["p_feas"],
            "p_opt": s["p_opt"],
        })
        return value

    demo_result = minimize(
        logged_objective,
        start,
        method="COBYLA",
        bounds=Bounds([0, 0], [2*np.pi, np.pi]),
        options={"maxiter": DEMO_BUDGET, "rhobeg": 0.5, "tol": 1e-8, "catol": 1e-8},
    )

cobyla_df = pd.DataFrame(cobyla_rows)
display(cobyla_df.head())

In [ ]:
# 6.3 Visualize what COBYLA receives and changes
if len(cobyla_df):
    plt.figure(figsize=(10, 4))
    plt.plot(cobyla_df["evaluation"], cobyla_df["objective"], alpha=0.6, label="objective evaluation")
    plt.plot(cobyla_df["evaluation"], cobyla_df["best_so_far"], linewidth=2.2, label="best so far")
    plt.xlabel("COBYLA evaluation")
    plt.ylabel(r"$\langle H_C\rangle$")
    plt.title("Scalar objective returned to COBYLA")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "09_cobyla_objective_trace.png", dpi=180, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(7, 6))
    sc = plt.scatter(
        cobyla_df["gamma_1"], cobyla_df["beta_1"],
        c=cobyla_df["evaluation"], s=36
    )
    plt.colorbar(sc, label="evaluation")
    plt.xlabel(r"$\gamma_1$")
    plt.ylabel(r"$\beta_1$")
    plt.title("COBYLA search path in parameter space")
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "10_cobyla_parameter_path.png", dpi=180, bbox_inches="tight")
    plt.show()

# 7. Main Results — Depth $p=1,\ldots,110$

We first read the frozen table

`results/global_depth110/depth_by_depth.csv`.

This is the authoritative quick path for the presentation.  
The expensive full optimizer rerun is provided later.

In [ ]:
# 7.1 Load and summarize frozen 110-depth sweep
depth_df = pd.read_csv(RESULT_ROOT / "depth_by_depth.csv")
assert len(depth_df) == 110
assert depth_df["p"].tolist() == list(range(1, 111))

penalty_best = depth_df.loc[depth_df["penalty_x_p_opt"].idxmax()]
grover_best = depth_df.loc[depth_df["global_grover_p_opt"].idxmax()]
cross = depth_df[depth_df["global_grover_p_opt"] > depth_df["penalty_x_p_opt"]]
first_cross = int(cross.iloc[0]["p"]) if len(cross) else None

print("Penalty-X best p       :", int(penalty_best["p"]))
print("Penalty-X best p_opt   :", penalty_best["penalty_x_p_opt"])
print("Global-Grover best p   :", int(grover_best["p"]))
print("Global-Grover best p_opt:", grover_best["global_grover_p_opt"])
print("First Grover crossover :", first_cross)

display(depth_df.tail())

In [ ]:
# 7.2 p_opt across depth
plt.figure(figsize=(11, 5))
plt.plot(depth_df["p"], depth_df["penalty_x_p_opt"], label="Penalty-X")
plt.plot(depth_df["p"], depth_df["global_grover_p_opt"], label="Global-Grover")
plt.xlabel("QAOA depth p")
plt.ylabel(r"$P_{\mathrm{opt}}$")
plt.title(r"Optimal-route probability, $p=1,\ldots,110$")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "11_depth_p_opt.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# 7.3 p_feas across depth
plt.figure(figsize=(11, 5))
plt.plot(depth_df["p"], depth_df["penalty_x_p_feas"], label="Penalty-X")
plt.plot(depth_df["p"], depth_df["global_grover_p_feas"], label="Global-Grover")
plt.xlabel("QAOA depth p")
plt.ylabel(r"$P_{\mathrm{feas}}$")
plt.title(r"Feasible probability mass, $p=1,\ldots,110$")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "12_depth_p_feas.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# 7.4 conditional p_opt and expected penalized energy
plt.figure(figsize=(11, 5))
plt.plot(
    depth_df["p"], depth_df["penalty_x_p_opt_given_feasible"],
    label="Penalty-X"
)
plt.plot(
    depth_df["p"], depth_df["global_grover_p_opt_given_feasible"],
    label="Global-Grover"
)
plt.xlabel("QAOA depth p")
plt.ylabel(r"$P_{\mathrm{opt}\mid\mathrm{feas}}$")
plt.title("Conditional concentration on the optimal route")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "13_depth_conditional_opt.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(
    depth_df["p"], depth_df["penalty_x_expected_penalized_cost"],
    label="Penalty-X"
)
plt.plot(
    depth_df["p"], depth_df["global_grover_expected_penalized_cost"],
    label="Global-Grover"
)
plt.xlabel("QAOA depth p")
plt.ylabel(r"$\langle H_C\rangle$")
plt.title("Optimized expected penalized energy")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "14_depth_expected_energy.png", dpi=180, bbox_inches="tight")
plt.show()

# 8. Evolution Microscope — Replay Every Operation at $p=110$

The frozen checkpoint contains 220 optimized parameters:

$$
(\gamma_1,\ldots,\gamma_{110},\beta_1,\ldots,\beta_{110}).
$$

We now replay

$$
|\psi_0\rangle
\to U_C(\gamma_1)
\to U_M(\beta_1)
\to \cdots
\to U_C(\gamma_{110})
\to U_M(\beta_{110})
$$

and record the quantum-state statistics after **every cost and mixer operation**.

In [ ]:
# 8.1 Load p=110 checkpoints
def load_checkpoint(algorithm, p):
    path = RESULT_ROOT / "checkpoints" / f"{algorithm}_p{p:03d}.json"
    return json.loads(path.read_text(encoding="utf-8"))

penalty_ckpt = load_checkpoint("penalty_x", 110)
grover_ckpt = load_checkpoint("global_grover", 110)

print("Penalty-X p110 evaluations:", penalty_ckpt["nfev"])
print("Grover p110 evaluations   :", grover_ckpt["nfev"])
print("Parameters per checkpoint :", len(penalty_ckpt["optimized_parameters"]))

In [ ]:
# 8.2 Layer-replay engine
def summarize_state(state):
    probs = probabilities(state)
    p_feas = float(probs[feasible_mask].sum())
    p_opt = float(probs[optimal_mask].sum())
    top = int(np.argmax(probs))
    s = states[top]

    return {
        "norm": float(np.linalg.norm(state)),
        "probability_sum": float(probs.sum()),
        "expected_hc": float(probs @ raw_energies),
        "expected_routing_term": float(probs @ routing_costs),
        "expected_flow_penalty": float(probs @ flow_penalties),
        "p_feas": p_feas,
        "p_opt": p_opt,
        "p_opt_given_feasible": p_opt / p_feas if p_feas > 0 else np.nan,
        "invalid_mass": 1 - p_feas,
        "entropy": shannon_entropy(probs),
        "max_basis_probability": float(probs[top]),
        "top_basis_index": top,
        "top_basis_label": s.canonical_bitstring,
        "top_is_feasible": bool(s.is_decoder_valid),
        "top_is_optimal": bool(optimal_mask[top]),
        "top_decoded_route": s.decoded_route,
    }

def replay_checkpoint(algorithm, checkpoint, n_energy_bins=48):
    p = int(checkpoint["depth"])
    theta = np.asarray(checkpoint["optimized_parameters"], dtype=float)
    gammas, betas = split_parameters(theta, p)

    bins = np.linspace(raw_energies.min(), raw_energies.max(), n_energy_bins + 1)
    centers = (bins[:-1] + bins[1:]) / 2

    state = initial_state(n_qubits)
    rows = [{
        "checkpoint_index": 0,
        "operation": "initial",
        "layer": 0,
        "gamma": np.nan,
        "beta": np.nan,
        "max_probability_change": 0.0,
        **summarize_state(state),
    }]

    histograms = []
    hist_layers = []
    h, _ = np.histogram(raw_energies, bins=bins, weights=probabilities(state))
    histograms.append(h)
    hist_layers.append(0)

    k = 0
    for layer, (gamma, beta) in enumerate(zip(gammas, betas), start=1):
        # COST
        before = probabilities(state)
        state = apply_cost(state, raw_energies, gamma)
        after = probabilities(state)
        k += 1
        rows.append({
            "checkpoint_index": k,
            "operation": "cost",
            "layer": layer,
            "gamma": gamma,
            "beta": beta,
            "max_probability_change": float(np.max(np.abs(after-before))),
            **summarize_state(state),
        })

        # MIXER
        before = after
        if algorithm == "penalty_x":
            state = apply_x_mixer(state, beta, n_qubits)  # no beta/n scaling
        elif algorithm == "global_grover":
            state = apply_grover_mixer(state, beta)
        else:
            raise ValueError(algorithm)

        after = probabilities(state)
        k += 1
        rows.append({
            "checkpoint_index": k,
            "operation": "mixer",
            "layer": layer,
            "gamma": gamma,
            "beta": beta,
            "max_probability_change": float(np.max(np.abs(after-before))),
            **summarize_state(state),
        })

        h, _ = np.histogram(raw_energies, bins=bins, weights=after)
        histograms.append(h)
        hist_layers.append(layer)

    return (
        pd.DataFrame(rows),
        np.asarray(hist_layers),
        centers,
        np.asarray(histograms),
        state,
    )

penalty_trace, penalty_hist_layers, energy_centers, penalty_hist, penalty_final = (
    replay_checkpoint("penalty_x", penalty_ckpt)
)
grover_trace, grover_hist_layers, _, grover_hist, grover_final = (
    replay_checkpoint("global_grover", grover_ckpt)
)

print("Trace rows per algorithm:", len(penalty_trace))
assert len(penalty_trace) == 221
assert len(grover_trace) == 221

In [ ]:
# 8.3 Exact replay validation against frozen checkpoint values
def validate_replay(name, trace, ckpt):
    final = trace.iloc[-1]
    pairs = [
        ("expected_hc", "expected_penalized_cost"),
        ("p_feas", "p_feas"),
        ("p_opt", "p_opt"),
        ("p_opt_given_feasible", "p_opt_given_feasible"),
    ]
    print(name)
    for trace_key, checkpoint_key in pairs:
        a = float(final[trace_key])
        b = float(ckpt[checkpoint_key])
        err = abs(a-b)
        print(f"{trace_key:24s} replay={a:.15g} frozen={b:.15g} error={err:.2e}")
        assert np.isclose(a, b, atol=2e-10, rtol=0)

    max_cost_delta = trace.loc[
        trace["operation"] == "cost", "max_probability_change"
    ].max()
    print("max probability change at any cost step:", max_cost_delta)
    assert max_cost_delta < 2e-12
    print("PASS\n")

validate_replay("Penalty-X p=110", penalty_trace, penalty_ckpt)
validate_replay("Global-Grover p=110", grover_trace, grover_ckpt)

In [ ]:
# 8.4 Cost-vs-mixer probability-change microscope
plt.figure(figsize=(12, 5))
plt.plot(penalty_trace["checkpoint_index"], penalty_trace["max_probability_change"])
plt.xlabel("operation checkpoint")
plt.ylabel("max basis-probability change")
plt.title("Penalty-X p=110: cost steps preserve probability, mixers change it")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "15_penalty_cost_mixer_probability_change.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(grover_trace["checkpoint_index"], grover_trace["max_probability_change"])
plt.xlabel("operation checkpoint")
plt.ylabel("max basis-probability change")
plt.title("Global-Grover p=110: cost steps preserve probability, mixers change it")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "16_grover_cost_mixer_probability_change.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# 8.5 One point per COMPLETED layer
penalty_layers = penalty_trace[penalty_trace["operation"].isin(["initial", "mixer"])]
grover_layers = grover_trace[grover_trace["operation"].isin(["initial", "mixer"])]

plt.figure(figsize=(11, 5))
plt.plot(penalty_layers["layer"], penalty_layers["expected_hc"], label="Penalty-X")
plt.plot(grover_layers["layer"], grover_layers["expected_hc"], label="Global-Grover")
plt.xlabel("completed layer")
plt.ylabel(r"$\langle H_C\rangle$")
plt.title("Energy after every completed QAOA layer")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "17_p110_layer_energy.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(penalty_layers["layer"], penalty_layers["p_feas"], label="Penalty-X")
plt.plot(grover_layers["layer"], grover_layers["p_feas"], label="Global-Grover")
plt.xlabel("completed layer")
plt.ylabel(r"$P_{\mathrm{feas}}$")
plt.title("Feasible probability after every completed layer")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "18_p110_layer_pfeas.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(penalty_layers["layer"], penalty_layers["p_opt"], label="Penalty-X")
plt.plot(grover_layers["layer"], grover_layers["p_opt"], label="Global-Grover")
plt.xlabel("completed layer")
plt.ylabel(r"$P_{\mathrm{opt}}$")
plt.title("Optimal-route probability after every completed layer")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "19_p110_layer_popt.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# 8.6 Conditional concentration and entropy
plt.figure(figsize=(11, 5))
plt.plot(
    penalty_layers["layer"], penalty_layers["p_opt_given_feasible"],
    label="Penalty-X"
)
plt.plot(
    grover_layers["layer"], grover_layers["p_opt_given_feasible"],
    label="Global-Grover"
)
plt.xlabel("completed layer")
plt.ylabel(r"$P_{\mathrm{opt}\mid\mathrm{feas}}$")
plt.title("Conditional concentration inside feasible routes")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "20_p110_conditional_opt.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(penalty_layers["layer"], penalty_layers["entropy"], label="Penalty-X")
plt.plot(grover_layers["layer"], grover_layers["entropy"], label="Global-Grover")
plt.xlabel("completed layer")
plt.ylabel("Shannon entropy [nats]")
plt.title("Distribution entropy during the optimized p=110 circuit")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "21_p110_entropy.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# 8.7 Optimized gamma_l and beta_l
def parameter_frame(ckpt):
    p = int(ckpt["depth"])
    theta = np.asarray(ckpt["optimized_parameters"], dtype=float)
    gamma, beta = split_parameters(theta, p)
    return pd.DataFrame({"layer": np.arange(1, p+1), "gamma": gamma, "beta": beta})

penalty_params = parameter_frame(penalty_ckpt)
grover_params = parameter_frame(grover_ckpt)

plt.figure(figsize=(11, 4))
plt.plot(penalty_params["layer"], penalty_params["gamma"], label=r"$\gamma_\ell$")
plt.plot(penalty_params["layer"], penalty_params["beta"], label=r"$\beta_\ell$")
plt.xlabel("layer")
plt.ylabel("angle [rad]")
plt.title("Penalty-X optimized p=110 parameters")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "22_penalty_p110_parameters.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 4))
plt.plot(grover_params["layer"], grover_params["gamma"], label=r"$\gamma_\ell$")
plt.plot(grover_params["layer"], grover_params["beta"], label=r"$\beta_\ell$")
plt.xlabel("layer")
plt.ylabel("angle [rad]")
plt.title("Global-Grover optimized p=110 parameters")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "23_grover_p110_parameters.png", dpi=180, bbox_inches="tight")
plt.show()

## 8.8 "Potential-well" style energy visualization

We bin basis states by penalized energy and compute

$$
P_\ell(E\in B_k)
=
\sum_{z:E_z\in B_k}P_\ell(z).
$$

This is **not** a literal spatial potential.  
It is a compact visualization of probability mass moving between energy ranges as QAOA evolves.

In [ ]:
# 8.8 Energy-depth heatmaps
import matplotlib as mpl
import matplotlib.patheffects as pe
from matplotlib.colors import PowerNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

optimal_energy = float(np.min(raw_energies[optimal_mask]))
positive_mass = np.concatenate([
    penalty_hist[penalty_hist > 0],
    grover_hist[grover_hist > 0],
])
shared_vmax = float(np.quantile(positive_mass, 0.995))
mass_norm = PowerNorm(gamma=0.50, vmin=0.0, vmax=shared_vmax)
mass_cmap = mpl.colormaps["Blues"].copy()
mass_cmap.set_bad("#f8fafc")
energy_bin_width = float(np.median(np.diff(energy_centers)))
energy_plot_min = min(
    optimal_energy - 0.35 * energy_bin_width,
    float(energy_centers.min()) - 0.5 * energy_bin_width,
)
energy_plot_max = float(energy_centers.max()) + 0.5 * energy_bin_width

def completed_layer_energy(trace):
    rows = trace.loc[
        (trace["operation"] == "initial") | (trace["operation"] == "mixer")
    ].sort_values("layer")
    return rows["layer"].to_numpy(), rows["expected_hc"].to_numpy()

heatmap_specs = [
    (penalty_hist_layers, penalty_hist, penalty_trace, "Penalty-X"),
    (grover_hist_layers, grover_hist, grover_trace, "Global-Grover"),
]

figure_bg = "#f3f6f8"
axes_bg = "#f8fafc"
curve_color = "#d1495b"
optimum_color = "#394b59"

with plt.style.context("default"):
    fig = plt.figure(figsize=(15.2, 6.4), facecolor=figure_bg)
    grid = fig.add_gridspec(
        1, 3, width_ratios=(1, 1, 0.035),
        left=0.07, right=0.94, bottom=0.16, top=0.78, wspace=0.10,
    )
    axes = [fig.add_subplot(grid[0, 0])]
    axes.append(fig.add_subplot(grid[0, 1], sharey=axes[0]))
    colorbar_ax = fig.add_subplot(grid[0, 2])

    for panel, ax, (layers, histogram, trace, title) in zip(
        ("a", "b"), axes, heatmap_specs
    ):
        visible_mass = np.ma.masked_less_equal(histogram.T, 0.0)
        image = ax.pcolormesh(
            layers, energy_centers, visible_mass,
            shading="nearest", cmap=mass_cmap, norm=mass_norm,
            rasterized=True, antialiased=False,
        )
        curve_layers, expected_energy = completed_layer_energy(trace)
        curve, = ax.plot(
            curve_layers, expected_energy, color=curve_color,
            linewidth=2.4, solid_capstyle="round", zorder=5,
        )
        curve.set_path_effects([
            pe.Stroke(linewidth=4.4, foreground="white", alpha=0.90),
            pe.Normal(),
        ])
        optimum = ax.axhline(
            optimal_energy, color=optimum_color, linewidth=1.6,
            linestyle=(0, (5, 3)), zorder=4,
        )
        optimum.set_path_effects([
            pe.Stroke(linewidth=3.0, foreground="white", alpha=0.80),
            pe.Normal(),
        ])
        ax.scatter(
            curve_layers[-1], expected_energy[-1], s=34,
            color=curve_color, edgecolor="white", linewidth=1.2, zorder=6,
        )

        ax.set_xlim(float(layers.min()), float(layers.max()))
        ax.set_ylim(energy_plot_min, energy_plot_max)
        ax.set_facecolor(axes_bg)
        ax.set_xlabel("Completed QAOA layer", labelpad=9)
        ax.set_title(
            f"{panel}   {title}", loc="left", pad=12,
            fontsize=13.5, fontweight="bold", color="#17212b",
        )
        ax.text(
            0.975, 0.955,
            rf"final $\langle H_C\rangle = {expected_energy[-1]:.1f}$",
            transform=ax.transAxes, ha="right", va="top", fontsize=9.5,
            color="#273744",
            bbox=dict(
                boxstyle="round,pad=0.35", facecolor="white",
                edgecolor="#d7dee5", alpha=0.92,
            ),
            zorder=7,
        )
        ax.grid(
            axis="x", color="#c8d1da", alpha=0.45,
            linewidth=0.7, linestyle=(0, (2, 3)), zorder=3,
        )
        ax.tick_params(colors="#40515f", labelsize=9.5)
        for spine in ax.spines.values():
            spine.set_color("#cbd5dd")
            spine.set_linewidth(0.8)

    axes[0].set_ylabel(r"Penalized energy $E$", labelpad=10)
    axes[1].tick_params(labelleft=False)

    colorbar = fig.colorbar(image, cax=colorbar_ax, extend="max")
    colorbar.set_label("Probability mass in energy bin", labelpad=12)
    colorbar.ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    colorbar.ax.tick_params(labelsize=9, colors="#40515f")
    colorbar.outline.set_edgecolor("#cbd5dd")

    legend_handles = [
        Line2D([0], [0], color=curve_color, linewidth=2.5,
               label=r"Expected energy $\langle H_C\rangle$"),
        Line2D([0], [0], color=optimum_color, linewidth=1.7,
               linestyle=(0, (5, 3)), label="Optimal feasible energy"),
    ]
    fig.legend(
        handles=legend_handles, loc="upper right", bbox_to_anchor=(0.94, 0.895),
        frameon=False, ncol=2, columnspacing=1.8, handlelength=2.8, fontsize=9.5,
    )
    fig.suptitle(
        "QAOA shifts probability toward lower-energy states",
        x=0.07, y=0.965, ha="left", fontsize=18, fontweight="bold",
        color="#17212b",
    )
    fig.text(
        0.07, 0.902,
        "Shared scale and identical energy bins make the two mixers directly comparable.",
        ha="left", fontsize=10.5, color="#526574",
    )
    fig.text(
        0.07, 0.055,
        "Power normalization (γ = 0.50); zero-mass bins are white; values above the 99.5th percentile are saturated.",
        ha="left", fontsize=9.2, color="#667987",
    )

    plt.savefig(
        OUTPUT_ROOT / "24_25_energy_depth_comparison.png",
        dpi=220, bbox_inches="tight", facecolor=figure_bg,
    )
    plt.show()

In [ ]:
# 8.9 3D energy-depth surface for Penalty-X
from matplotlib import colormaps
from matplotlib.colors import Normalize

probability_floor = 1e-8
optimal_energy = float(np.min(raw_energies[optimal_mask]))

penalty_completed = penalty_trace.loc[
    (penalty_trace["operation"] == "initial")
    | (penalty_trace["operation"] == "mixer")
].sort_values("layer")

# Crop empty high-energy space using the time-averaged distribution. Keep the
# optimum and the complete expected-energy trajectory inside the frame.
time_averaged_mass = penalty_hist.mean(axis=0)
energy_cdf = np.cumsum(time_averaged_mass) / time_averaged_mass.sum()
lo = max(0, int(np.searchsorted(energy_cdf, 0.002)) - 1)
hi = min(len(energy_centers) - 1, int(np.searchsorted(energy_cdf, 0.998)) + 1)
important_energies = np.r_[
    optimal_energy, penalty_completed["expected_hc"].to_numpy(),
]
lo = min(lo, int(np.searchsorted(energy_centers, important_energies.min())))
hi = max(hi, int(np.searchsorted(energy_centers, important_energies.max())))
lo = max(0, lo - 1)
hi = min(len(energy_centers) - 1, hi + 1)

energy_plot = energy_centers[lo:hi + 1]
mass_plot = penalty_hist[:, lo:hi + 1].T
layer_mesh, energy_mesh = np.meshgrid(penalty_hist_layers, energy_plot)
log_mass = np.log10(np.clip(mass_plot, probability_floor, None))
z_floor = float(np.log10(probability_floor))
z_ceiling = 0.0
surface_mass = np.ma.masked_where(mass_plot <= probability_floor, log_mass)

background = "#07131b"
foreground = "#e8f0f3"
muted = "#9eb2bc"
trajectory_color = "#55e6d1"
optimum_color = "#ffd166"
cmap = colormaps["magma"]
color_norm = Normalize(vmin=z_floor, vmax=z_ceiling)

with plt.style.context("dark_background"), plt.rc_context({
    "font.size": 10.5,
    "axes.labelcolor": foreground,
    "xtick.color": muted,
    "ytick.color": muted,
}):
    fig = plt.figure(figsize=(14.5, 8.2), facecolor=background)
    ax = fig.add_subplot(111, projection="3d", facecolor=background)
    ax.set_proj_type("ortho")

    # A restrained floor projection preserves the global flow pattern, while
    # masking the numerical floor removes the large flat sheet from the surface.
    floor_colors = cmap(color_norm(log_mass))
    floor_colors[..., 3] = 0.32
    ax.plot_surface(
        layer_mesh, energy_mesh, np.full_like(log_mass, z_floor),
        facecolors=floor_colors, shade=False, linewidth=0, antialiased=False,
        rcount=len(energy_plot), ccount=len(penalty_hist_layers), zorder=0,
    )
    surface = ax.plot_surface(
        layer_mesh, energy_mesh, surface_mass,
        cmap=cmap, norm=color_norm, linewidth=0, shade=True,
        antialiased=True, rcount=len(energy_plot),
        ccount=len(penalty_hist_layers), alpha=0.94, zorder=2,
    )

    trajectory_z = np.full(len(penalty_completed), z_floor + 0.10)
    ax.plot(
        penalty_completed["layer"], penalty_completed["expected_hc"],
        trajectory_z, color=background, linewidth=5.2, zorder=8,
    )
    ax.plot(
        penalty_completed["layer"], penalty_completed["expected_hc"],
        trajectory_z, color=trajectory_color, linewidth=2.5,
        label=r"expected energy $\langle H_C\rangle$", zorder=9,
    )
    ax.plot(
        [penalty_hist_layers.min(), penalty_hist_layers.max()],
        [optimal_energy, optimal_energy], [z_floor + 0.12, z_floor + 0.12],
        color=optimum_color, linewidth=2.0, linestyle=(0, (5, 4)),
        label="optimal feasible energy", zorder=9,
    )

    ax.set(
        xlim=(float(penalty_hist_layers.min()), float(penalty_hist_layers.max())),
        ylim=(float(energy_plot.min()), float(energy_plot.max())),
        zlim=(z_floor, z_ceiling + 0.18),
        xlabel="completed QAOA layer",
        ylabel=r"penalized energy $E$",
        zlabel="probability mass per bin",
    )
    ax.xaxis.labelpad = 13
    ax.yaxis.labelpad = 13
    ax.zaxis.labelpad = 10
    ax.view_init(elev=27, azim=-124)
    ax.set_box_aspect((2.05, 1.0, 0.72))

    z_ticks = np.arange(z_floor, z_ceiling + 0.1, 2.0)
    ax.set_zticks(z_ticks)
    ax.set_zticklabels([rf"$10^{{{int(t)}}}$" for t in z_ticks])
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((0.035, 0.075, 0.10, 0.42))
        axis.pane.set_edgecolor((0.65, 0.76, 0.81, 0.16))
        axis._axinfo["grid"].update(
            color=(0.68, 0.78, 0.82, 0.14), linewidth=0.55, linestyle="-",
        )
    ax.tick_params(pad=2, labelsize=9.5)

    legend = ax.legend(
        loc="upper left", bbox_to_anchor=(0.02, 0.93),
        frameon=True, facecolor=background, edgecolor=(1, 1, 1, 0.18),
        framealpha=0.90, fontsize=9.5,
    )
    for text in legend.get_texts():
        text.set_color(foreground)

    colorbar = fig.colorbar(
        surface, ax=ax, shrink=0.62, pad=0.035, aspect=26,
        ticks=z_ticks,
    )
    colorbar.ax.set_yticklabels([rf"$10^{{{int(t)}}}$" for t in z_ticks])
    colorbar.set_label("probability mass per energy bin", color=foreground, labelpad=10)
    colorbar.outline.set_edgecolor((1, 1, 1, 0.18))
    colorbar.ax.tick_params(colors=muted)

    fig.suptitle(
        "Penalty-X probability flow", x=0.48, y=0.955,
        fontsize=19, fontweight="bold", color=foreground,
    )
    fig.text(
        0.48, 0.905,
        "Energy-resolved state probability across 110 completed QAOA layers",
        ha="center", fontsize=10.5, color=muted,
    )
    fig.text(
        0.48, 0.035,
        r"Log scale; mass $\leq 10^{-8}$ is omitted from the raised surface.",
        ha="center", fontsize=9.5, color=muted,
    )
    fig.subplots_adjust(left=0.015, right=0.91, bottom=0.075, top=0.90)
    plt.savefig(
        OUTPUT_ROOT / "26_penalty_energy_depth_3d.png",
        dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor(),
    )
    plt.show()

In [ ]:
# 8.10 Top final basis states
print("Penalty-X p=110 top states")
display(pd.DataFrame(
    top_state_rows(probabilities(penalty_final), states, optimal_cost=optimal_cost, top_k=15)
))

print("Global-Grover p=110 top states")
display(pd.DataFrame(
    top_state_rows(probabilities(grover_final), states, optimal_cost=optimal_cost, top_k=15)
))

# 9. Why Grover Looks Better but Still Loses

For a feasible optimum,

$$
\boxed{
P_{\rm opt}
=
P_{\rm feas}
P_{\rm opt\mid feas}
}.
$$

So Global-Grover can have a stronger conditional concentration on the optimum while still carrying too little total probability inside the feasible subset.

In [ ]:
# 9.1 Decompose p=110 success
r110 = depth_df.loc[depth_df["p"] == 110].iloc[0]

decomposition = pd.DataFrame([
    {
        "method": "Penalty-X",
        "P_feas": r110["penalty_x_p_feas"],
        "P_opt_given_feas": r110["penalty_x_p_opt_given_feasible"],
        "P_feas * P_opt_given_feas": (
            r110["penalty_x_p_feas"] * r110["penalty_x_p_opt_given_feasible"]
        ),
        "P_opt": r110["penalty_x_p_opt"],
    },
    {
        "method": "Global-Grover",
        "P_feas": r110["global_grover_p_feas"],
        "P_opt_given_feas": r110["global_grover_p_opt_given_feasible"],
        "P_feas * P_opt_given_feas": (
            r110["global_grover_p_feas"] * r110["global_grover_p_opt_given_feasible"]
        ),
        "P_opt": r110["global_grover_p_opt"],
    }
])
display(decomposition)

assert np.allclose(
    decomposition["P_feas * P_opt_given_feas"],
    decomposition["P_opt"]
)

cond_wins = int(
    (depth_df["global_grover_p_opt_given_feasible"] >
     depth_df["penalty_x_p_opt_given_feasible"]).sum()
)
feas_wins = int(
    (depth_df["penalty_x_p_feas"] >
     depth_df["global_grover_p_feas"]).sum()
)

print(f"Global-Grover has higher P(opt|feas) at {cond_wins}/110 depths.")
print(f"Penalty-X has higher P(feas) at {feas_wins}/110 depths.")

# 10. Scientific Computing & Validation

The interpretation is accepted only if the entire chain agrees:

$$
\text{graph}
\rightarrow
\text{QUBO}
\rightarrow
\text{Ising}
\rightarrow
\text{statevector}
\rightarrow
\text{probability}
\rightarrow
\text{decoded route}.
$$

In [ ]:
# 10.1 Final validation table
checks = []

checks.append(("Exact path cost", path_cost(graph, optimal_route), path_cost(graph, optimal_route) == optimal_cost))
checks.append(("Feasible states", int(feasible_mask.sum()), int(feasible_mask.sum()) == 20))
checks.append(("Optimal states", int(optimal_mask.sum()), int(optimal_mask.sum()) == 1))
checks.append(("QUBO-Ising max error", ising_error, ising_error == 0.0))
checks.append(("Uniform state norm", np.linalg.norm(initial_state(n_qubits)), np.isclose(np.linalg.norm(initial_state(n_qubits)), 1)))
checks.append(("Cost probability invariant", cost_probability_change, cost_probability_change < 1e-12))
checks.append((
    "Penalty-X p110 replay p_opt error",
    abs(float(penalty_trace.iloc[-1]["p_opt"]) - float(penalty_ckpt["p_opt"])),
    np.isclose(float(penalty_trace.iloc[-1]["p_opt"]), float(penalty_ckpt["p_opt"]), atol=2e-10, rtol=0)
))
checks.append((
    "Grover p110 replay p_opt error",
    abs(float(grover_trace.iloc[-1]["p_opt"]) - float(grover_ckpt["p_opt"])),
    np.isclose(float(grover_trace.iloc[-1]["p_opt"]), float(grover_ckpt["p_opt"]), atol=2e-10, rtol=0)
))

validation_df = pd.DataFrame(checks, columns=["check", "value", "pass"])
display(validation_df)
assert validation_df["pass"].all()
print("ALL VALIDATIONS PASS")

In [ ]:
# 10.2 Frozen p=110 runtime profile
runtime_df = pd.DataFrame([
    {
        "method": "Penalty-X",
        "optimizer_wall_time_s": penalty_ckpt["optimizer_wall_time_s"],
        "total_wall_time_s": penalty_ckpt["wall_time_s"],
        "nfev": penalty_ckpt["nfev"],
        "termination": penalty_ckpt["termination_reason"],
    },
    {
        "method": "Global-Grover",
        "optimizer_wall_time_s": grover_ckpt["optimizer_wall_time_s"],
        "total_wall_time_s": grover_ckpt["wall_time_s"],
        "nfev": grover_ckpt["nfev"],
        "termination": grover_ckpt["termination_reason"],
    },
])
display(runtime_df)

# 11. Optional Full Re-Optimization of $p=1,\ldots,110$

The frozen checkpoints are the recommended path for a presentation.

If you want to recompute the entire optimizer sweep, set

```python
RUN_FULL_110_OPTIMIZATION = True
```

below.

The rerun implements the frozen policy:

$$
\text{seed}_p=2601+7919p,
$$

$$
B_p=\max(120,4p+64),
$$

and for $p>1$ it continues from the optimized $p-1$ angles plus one new $\gamma,\beta\in[0,0.05]$.

**This can take a long time.**

In [ ]:
# 11.1 Full sweep reproduction utilities
RUN_FULL_110_OPTIMIZATION = False

class BudgetExhausted(RuntimeError):
    pass

def depth_seed(p):
    return 2601 + 7919 * int(p)

def depth_budget(p):
    return max(120, 4 * int(p) + 64)

def continuation_start(previous, p, seed):
    if p == 1:
        return initial_parameters(1, seed)

    previous = np.asarray(previous, dtype=float)
    prev_p = p - 1
    prev_gamma = previous[:prev_p]
    prev_beta = previous[prev_p:]

    rng = np.random.default_rng(seed)
    new_gamma = rng.uniform(0.0, 0.05)
    new_beta = rng.uniform(0.0, 0.05)

    return np.concatenate([prev_gamma, [new_gamma], prev_beta, [new_beta]])

def optimize_with_start(start, p, algorithm, budget):
    lower = np.array([0.0]*p + [0.0]*p)
    upper = np.array([2*np.pi]*p + [np.pi]*p)

    evaluations = 0
    best_value = np.inf
    best_theta = np.asarray(start, dtype=float).copy()
    initial_value = None

    def counted(theta):
        nonlocal evaluations, best_value, best_theta, initial_value
        if evaluations >= budget:
            raise BudgetExhausted

        evaluations += 1
        theta = np.asarray(theta, dtype=float)

        outside = (
            np.maximum(lower-theta, 0).sum()
            + np.maximum(theta-upper, 0).sum()
        )
        value = 1_000_000 + outside if outside else objective(theta, p, algorithm)

        if initial_value is None:
            initial_value = float(value)

        if not outside and value < best_value:
            best_value = float(value)
            best_theta = theta.copy()

        return value

    started = time.perf_counter()
    result = None

    try:
        result = minimize(
            counted,
            start,
            method="COBYLA",
            bounds=Bounds(lower, upper),
            options={
                "maxiter": budget,
                "rhobeg": 0.5,
                "tol": 1e-8,
                "catol": 1e-8,
            },
        )
    except BudgetExhausted:
        pass

    wall = time.perf_counter() - started

    if result is not None:
        candidate = np.asarray(result.x, dtype=float)
        if (
            np.all(candidate >= lower)
            and np.all(candidate <= upper)
            and float(result.fun) < best_value
        ):
            best_theta = candidate
            best_value = float(result.fun)

    return {
        "parameters": best_theta,
        "objective": best_value,
        "initial_objective": initial_value,
        "nfev": evaluations,
        "wall_time_s": wall,
    }

def rerun_depth_sweep(algorithm):
    rows = []
    previous = None

    save_dir = OUTPUT_ROOT / "recomputed_checkpoints"
    save_dir.mkdir(parents=True, exist_ok=True)

    for p in range(1, 111):
        seed = depth_seed(p)
        budget = depth_budget(p)
        start = continuation_start(previous, p, seed)

        result = optimize_with_start(start, p, algorithm, budget)
        previous = np.asarray(result["parameters"])

        final_state = simulate_full_space(previous, p, algorithm)
        summary = state_summary(final_state)

        row = {
            "algorithm": algorithm,
            "p": p,
            "seed": seed,
            "budget": budget,
            "nfev": result["nfev"],
            "wall_time_s": result["wall_time_s"],
            **summary,
        }
        rows.append(row)

        payload = {
            **row,
            "initial_parameters": start.tolist(),
            "optimized_parameters": previous.tolist(),
        }
        (save_dir / f"{algorithm}_p{p:03d}.json").write_text(
            json.dumps(payload, indent=2),
            encoding="utf-8"
        )

        print(
            f"{algorithm:14s} p={p:3d} "
            f"E={summary['expected_hc']:.6f} "
            f"p_feas={summary['p_feas']:.6f} "
            f"p_opt={summary['p_opt']:.8f} "
            f"nfev={result['nfev']}"
        )

    return pd.DataFrame(rows)

if RUN_FULL_110_OPTIMIZATION:
    recomputed_penalty = rerun_depth_sweep("penalty_x")
    recomputed_grover = rerun_depth_sweep("global_grover")

    recomputed_penalty.to_csv(
        OUTPUT_ROOT / "recomputed_penalty_x_depth110.csv", index=False
    )
    recomputed_grover.to_csv(
        OUTPUT_ROOT / "recomputed_global_grover_depth110.csv", index=False
    )
else:
    print("Full p=1..110 re-optimization is disabled.")
    print("The notebook still exactly replays the frozen p=110 optimized circuits.")

# 12. Takeaways and Limitations

### Takeaway 1 — the model defines the energy landscape

$$
\text{graph}\rightarrow Q(x)\rightarrow H_C.
$$

### Takeaway 2 — the cost layer writes energy into phase

$$
U_C(\gamma)|z\rangle=e^{-i\gamma E_z}|z\rangle.
$$

It does not directly change $P(z)$.

### Takeaway 3 — the mixer changes probability through interference

Penalty-X and Global-Grover couple amplitudes in very different ways.

### Takeaway 4 — COBYLA is a classical outer loop

It searches $(\boldsymbol\gamma,\boldsymbol\beta)$ using only scalar objective values.

### Takeaway 5 — depth is not monotonic success

The frozen $p=1,\ldots,110$ trajectories are oscillatory and all cells are subject to finite optimizer budgets.

### Takeaway 6 — conditional concentration is not global success

$$
P_{\rm opt}
=
P_{\rm feas}P_{\rm opt\mid feas}.
$$

### Limitations

- one fixed 7-node / 14-edge graph,
- ideal exact NumPy statevector simulation,
- no hardware noise,
- finite COBYLA evaluation budgets,
- full-space Global-Grover is the repository's specific projector convention,
- no claim of quantum advantage.

In [ ]:
# Final summary for presentation
print("Figures saved to:", OUTPUT_ROOT)

print(
    f"Penalty-X best: p={int(penalty_best['p'])}, "
    f"p_opt={penalty_best['penalty_x_p_opt']:.8g}"
)
print(
    f"Global-Grover best: p={int(grover_best['p'])}, "
    f"p_opt={grover_best['global_grover_p_opt']:.8g}"
)
print("First Global-Grover crossover:", first_cross)
print()

display(decomposition)